# 04. Proximity Buffers & Location Sensitivity Diagnostics

Evaluates revenue anomalies across concentric geofence rings (`0-500m`, `500-1500m`, `1500-5000m`, `no_incident`) and inspects distance-decay sensitivity.
Distance-based ring diagnostics require exact storefront coordinates (`coords_verified=True`) before production sign-off.

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path(os.environ.get("BGA_REPO_ROOT", Path.cwd())).resolve()
if not (REPO_ROOT / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/bga-matplotlib")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import load_config
from src.osint_collector import load_incidents_file
from src.spatial_models import (
    distance_decay_weight,
    enrich_incidents_spatial,
    join_sales_with_incidents,
    summarize_geofence_sensitivity,
)

DATA_MODE = os.environ.get("BGA_DATA_MODE", "sample").strip().lower()
cfg = load_config(repo_root=REPO_ROOT)
print(f"Store coords=({cfg.lat}, {cfg.lon}) | coords_verified={cfg.coords_verified} | radii_m={cfg.geofence_radii_m}")

In [ ]:
if DATA_MODE == "raw":
    from src.db_loader import load_daily_sales
    daily_df = load_daily_sales(config=cfg)
else:
    daily_df = pd.read_csv(cfg.sample_dir / "daily_sales.csv", parse_dates=["date"])

incidents_df = load_incidents_file(cfg.sample_dir / "incidents.csv", relevant_only=True)
enriched_inc = enrich_incidents_spatial(
    incidents_df, store_lat=cfg.lat, store_lon=cfg.lon, radii_m=cfg.geofence_radii_m
)
print("Incident counts by geofence ring:")
print(enriched_inc["geofence_ring"].value_counts().to_string())

joined = join_sales_with_incidents(
    sales_df=daily_df,
    incidents_df=incidents_df,
    store_lat=cfg.lat,
    store_lon=cfg.lon,
    radii_m=cfg.geofence_radii_m,
    max_distance_m=None,
)
ring_summary = summarize_geofence_sensitivity(joined)
print("\nGeofence Sensitivity Table:")
print(ring_summary.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(ring_summary["ring"], ring_summary["mean_residual_pct"] * 100.0, color="#2b8cbe")
axes[0].axhline(0.0, color="black", linestyle="--", linewidth=0.8)
axes[0].set_title("Mean Sales Residual (%) by Geofence Ring")
axes[0].tick_params(axis="x", rotation=20)

d_grid = np.linspace(0, 6000, 120)
axes[1].plot(d_grid, distance_decay_weight(d_grid, half_life_m=1500.0), label="Exponential (half-life 1500m)")
axes[1].plot(d_grid, distance_decay_weight(d_grid, half_life_m=1500.0, kernel="inverse"), linestyle="--", label="Inverse Distance")
axes[1].set_title("Distance-Decay Weight Curves")
axes[1].set_xlabel("Distance from Storefront (m)")
axes[1].legend()
plt.tight_layout()
plt.close(fig)